### After clearing gates:
1. Data Governance: Must data stay private & never leave machine?
    - self hosted/No API\n
2. Cost Cieling: Zero budget -> every paid API is out.
3. hardware fit --> Does it actually run on my device?
    - Too big to load = disqualified, not "low quality"

The selected model needs to be scored based on:
1. faithfulness
2. Grounding
3. Refusal behaviour
4. Citation / tracibility
5. Controllability and latency


### Selected models

1. Llama 3.2 3B Instruct	3B	128K	Meta Llama Community License
2. Qwen2.5-3B-Instruct	3B	32K	Qwen custom license
3. Microsoft Phi-4-mini-instruct	3.8B	128K	MIT	Strongest small option for reasoning and document QA
4. Ministral 3B Instruct	3B	128K	Mistral Research/custom license
5. IBM Granite 3.3 2B Instruct	2B	128K	Apache 2.0	Good long-context and enterprise-document option

> The above model still needs to be calculate dfor the how much memory they will occupy during runtime and overhead because of kv caching


## # Creating a function to calculate the KV cache size overhead size and the estimate of the model size that fir in the model or not

In [ ]:
import os
os.chdir("./..")

In [ ]:
## I need some component details of the architecture to calculate th =e kv cache size
# Number of kv heads
# Number of layers
# The output dimenion of the model
import requests
from huggingface_hub import ModelCard, hf_hub_download
import json
def pull_config(model_id):
    url = f"https://huggingface.co/{model_id}/resolve/main/config.json"
    cont = requests.get(url)
    if cont.status_code == 200:
        content = requests.get(url).json()
    else:
        path = hf_hub_download(
                    repo_id=model_id,
                    filename="config.json"
                    )
        content = json.load(open(path))
    mechanics = {"model": model_id.split("/")[1],
                    "context_window": content.get('max_position_embeddings',""),
                    "precision": content.get('torch_dtype',""),
                    "specs": {"num_layers": content.get("num_hidden_layers",""),
                            "num_kv_heads": content.get("num_key_value_heads",""),
                            "num_attention_heads": content.get("num_attention_heads",""),
                            "head_size": content.get("hidden_size","")}
                            }
    
        
    return mechanics

pull_config("meta-llama/Llama-3.2-3B-Instruct")

    

In [ ]:
def calculate_kv_cache_size(model_id, token_sequence_length):
    model_specs = pull_config(model_id)
    if model_specs['precision'] == 'bfloat16' or model_specs['dtype'] == 'bfloat16':
        bytes_per_number = 2
    elif model_specs['precision'] == 'F32' or model_specs['precision'] == 'FP32':
        bytes_per_number = 4
    elif model_specs['precision'] == 'F8' or model_specs['precision'] == 'FP8':
        bytes_per_number = 1
    else:
        print(model_specs['precision'])
        raise ValueError(f"Unknown precision: {model_specs['precision']}")

    num_kv_heads = model_specs['specs']['num_kv_heads']
    num_layers = model_specs['specs']['num_layers']
    head_dim = model_specs['specs']['head_size']/model_specs['specs']['num_attention_heads']

    peak_kv = 2*num_kv_heads*num_layers*head_dim*bytes_per_number*token_sequence_length

    kv_in_MB = peak_kv / (1024*1024)

    return f"{kv_in_MB} MB", model_specs

In [ ]:
calculate_kv_cache_size("ibm-granite/granite-3.3-2b-instruct", token_sequence_length=8000)

In [ ]:
#from sentence_transformers import SentenceTransformer
#model = SentenceTransformer('BAAI/bge-base-en-v1.5')


In [ ]:
#total_params = sum(p.numel() for p in model.parameters())

In [ ]:
# check precision
'''for p in model.parameters():
    print(p.dtype)
    break'''

### Now we know the embedding model has 110M params and FP32 precision

In [ ]:
'''# Calculate the size of embedding model
byte_per_numner = 4
embedder_size = (total_params * 4)/(1024*1024)
print(f"{embedder_size} MB")'''

### Create a estimator function
* I am going to use ollama and Q4 quantised model which is default so the weights precision will be `4-bit` and byte per number will be `0.5`

In [ ]:
def estimate_memory_usage(model_id, model_size_from_ollama, token_sequence_length, embedder_size = 417.6416, overhead = 1000):
    kv_cache_size, model_spec = calculate_kv_cache_size(model_id, token_sequence_length)
    cache_size = float(kv_cache_size.split()[0])/1024
    
    Total_estimated_memory_usage = cache_size + embedder_size/1024 + model_size_from_ollama + overhead/1024

    total_usage_in_GB = Total_estimated_memory_usage
    return f"{total_usage_in_GB} GB", model_spec



In [ ]:
granite33_2b = estimate_memory_usage("ibm-granite/granite-3.3-2b-instruct",model_size_from_ollama=1.5,token_sequence_length=8000)
qwen25_3b = estimate_memory_usage("Qwen/Qwen2.5-3B-Instruct", model_size_from_ollama=1.9,token_sequence_length=8000)
phi4 = estimate_memory_usage("microsoft/Phi-4-mini-instruct",model_size_from_ollama=2.5,token_sequence_length=8000)
llama = estimate_memory_usage("meta-llama/Llama-3.2-3B-Instruct", model_size_from_ollama=2,token_sequence_length=8000)


In [ ]:
results_8000 = [granite33_2b, qwen25_3b, phi4, llama]

In [ ]:
granite33_2b_6 = estimate_memory_usage("ibm-granite/granite-3.3-2b-instruct",model_size_from_ollama=1.5,token_sequence_length=6000)
qwen25_3b_6 = estimate_memory_usage("Qwen/Qwen2.5-3B-Instruct", model_size_from_ollama=1.9,token_sequence_length=6000)
phi4_6 = estimate_memory_usage("microsoft/Phi-4-mini-instruct",model_size_from_ollama=2.5,token_sequence_length=6000)
llama_6 = estimate_memory_usage("meta-llama/Llama-3.2-3B-Instruct", model_size_from_ollama=2,token_sequence_length=6000)


In [ ]:
results_6000 = [granite33_2b_6, qwen25_3b_6 ,phi4_6 ,llama_6]

In [ ]:
granite33_2b_4 = estimate_memory_usage("ibm-granite/granite-3.3-2b-instruct",model_size_from_ollama=1.5,token_sequence_length=4000)
qwen25_3b_4 = estimate_memory_usage("Qwen/Qwen2.5-3B-Instruct", model_size_from_ollama=1.9,token_sequence_length=4000)
phi4_4 = estimate_memory_usage("microsoft/Phi-4-mini-instruct",model_size_from_ollama=2.5,token_sequence_length=4000)
llama_4 = estimate_memory_usage("meta-llama/Llama-3.2-3B-Instruct", model_size_from_ollama=2,token_sequence_length=4000)


In [ ]:
results_4000 = [granite33_2b_4, qwen25_3b_4 ,phi4_4 ,llama_4]

In [ ]:
data = []
def add_results(results, sequence_length):
    for memory, info in results:
        specs = info["specs"]

        data.append({
            "model": info["model"],
            "sequence_length": sequence_length,
            "memory_gb": float(memory.split()[0]),
            "context_window": info["context_window"],
            "precision": info["precision"],
            "num_layers": specs["num_layers"],
            "num_kv_heads": specs["num_kv_heads"],
            "num_attention_heads": specs["num_attention_heads"],
            "head_size": specs["head_size"]
        })


add_results(results_8000, 8000)
add_results(results_6000,6000)
add_results(results_4000,4000)

In [ ]:
import pandas as pd
df = pd.DataFrame(data)
df

> I am keeping ministral out of the table for now.
* Phi4 is exceeding the memory even at 6000 sequence length.
* Llama for 8000 token sequence,  it was exceeding to 4.2 and for 6000 length 4.02. But it's because 1 GB of overhead. I think we will try each model for scoring.


## Create a function to run OLLAMA

`!ollama pull granite3.3:2b` I ran this command in terminal to download the model.

In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "granite3.3:2b",
        "prompt": "what is a medical device?",
        "stream": False
    }
)

In [ ]:
print(response.json())

In [ ]:
def ask_ollama(model_name:str, prompt: str):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model":model_name,
                "prompt": prompt,
                "stream": False
            }
        )
    except Exception as e:
        return f"[REQUEST FAILED] could not reach Ollama: {e}"
    data = response.json()  
    print(data.get("prompt_eval_count"), data.get("eval_count"))
    if response.status_code == 200:
        return data.get("response", "[NO RESPONSE FIELD IN REPLY]")
    else:
        return f"[OLLAMA ERROR {response.status_code}] {data.get('error', 'unknown error')}"

In [ ]:
#ask_ollama(model_name="granite3.3:2b", prompt="What is the capital of India?")

## Building a Prompt for the model

### Create or get a client

In [ ]:
!pwd

In [ ]:
import chromadb

# define the database
client = chromadb.PersistentClient("data/database/chromadb/")

# get the collection
collection = client.get_collection(name='mdr')
collection.count()

### Create a Prompt-assembly function

In [ ]:
# Create a function that build source of the chunk

def create_source(metadata: dict):

    def label(number, title):
        if title:
            return f"{number} ({title})"
        return number
    
    article= metadata.get('article', "").strip()
    annex = metadata.get("annex","").strip()
    chapter = str(metadata.get('chapter',"")).strip()
    chapter_title = metadata.get("chapter_title", "").strip()
    annex_title = metadata.get("annex_title","").strip()
    article_title = metadata.get("article_title", "").strip()
    section = metadata.get("section","").strip()
    section_title = metadata.get("section_title").strip()

    if chapter == "0" and chapter_title == "preamble":
        source = "EU MDR Preamble"
    
    elif article:
        parts = []
        if chapter: parts.append(label(chapter, chapter_title))
        if section: parts.append(label(section,section_title))
        parts.append(label(article, article_title))
        source =  ", ".join(parts)

    elif annex:
        parts = [label(annex, annex_title)]
        if chapter: parts.append(label(chapter, chapter_title))
        source =  ", ".join(parts)

    return source



In [ ]:
# Building a function to create the complete context of the model

def create_model_context(retrieved_chunks: dict) -> str:
    list_of_chunks = []
    docs = retrieved_chunks['documents'][0]
    metadatas = retrieved_chunks['metadatas'][0]
    for content, meta in zip(docs, metadatas):
        source = create_source(meta)
        page_c = "<CHUNK_SOURCE: " + source + ">\n" + content + "</CHUNK>"
        list_of_chunks.append(page_c)
    
    context = "\n-------------------------------------------------------------------------------------------------\n".join(list_of_chunks)
    return context

In [ ]:
def create_prompt(md_file, question, Example = None):
    results = collection.query(
        query_texts=[question],
        n_results=5
    )
    model_context = create_model_context(results)
    with open(md_file, 'r') as f:
        prompt = f.readlines()
    if Example:
        prompt.insert(len(prompt)-1, f"<EXAMPLE:>\n{Example} \n</EXAMPLE>\n")
    prompt.insert(len(prompt)-1, f"<SOURCES:>\n{model_context} \n</SOURCES>\n")
    prompt.insert(len(prompt)-1, f"<QUESTION:>\n{question} \n</QUESTION>\n")

    return "".join(prompt)


In [ ]:
#prompt = create_prompt("prompt.md", "what is medical device?")

In [ ]:
#ask_ollama("granite3.3:2b", prompt=prompt)

In [ ]:
example = """
USER:
How do I know when I need to submit a trend report for minor incidents or expected side effects, and what do I need to include in my plan?
SPECIALIST:
[Article 88] You must submit a trend report via the electronic system when there is a statistically significant increase in the frequency or severity of non-serious incidents or expected undesirable side effects.
[Article 88] This applies if the increase could significantly impact the device's benefit-risk analysis and lead to unacceptable risks to patient or user safety when weighed against intended benefits.
[Article 88(1)] You must compare occurrences against the foreseeable frequency or severity established in your technical documentation and product information for that device or device group over a specified timeframe.
[Article 88(1)] Your post-market surveillance plan must document how you will manage these non-serious incidents and side effects, the methodology used to calculate statistically significant increases, and the observation period for monitoring trends.
[Article 92] The trend report is submitted through the electronic system.
[Article 88(2)] Competent authorities may review your trend reports and require you to adopt appropriate corrective measures to protect public health and safety.

KEY TAKEAWAY: 
A trend report is required when non-serious incidents or expected side effects rise significantly enough to affect the device's benefit-risk balance [Article 88]. Your PMS plan must document the management approach, the method for calculating significant increases, and the observation period [Article 88(1)], and competent authorities may require corrective measures based on your reports [Article 88(2)].

USER:
What is the definition of an 'importer' under this Regulation?
SPECIALIST:
The answer is not present in the provided document.

KEY TAKEAWAY:
The provided sources describe obligations and procedures involving importers but do not contain the definition of 'importer'. No answer can be given from the available context.
"""

In [ ]:
prompt = create_prompt("prompt.md", "What is medical device?", Example=example)
#print(prompt)

In [ ]:
prompt_2 = create_prompt("prompt.md", "What are the general obligations of the 'manufacturers' to place device in the market?", example)
#print(prompt_2)

In [ ]:
prompt_3 = create_prompt("prompt.md", "What all to consider while puting custom made devices in the market?", example)
#print(prompt_3)

In [ ]:
prompt_4 = create_prompt("prompt.md","what is 'notified body'?", example)
#print(prompt_4)

### Testing with granite3.3:2b

In [ ]:
print(ask_ollama("granite3.3:2b", prompt))

In [ ]:
print(ask_ollama("granite3.3:2b", prompt_2))

In [ ]:
print(ask_ollama("granite3.3:2b", prompt_3))

In [ ]:
print(ask_ollama("granite3.3:2b", prompt_4))

### Testing with  qwen2.5:3b

!ollama pull qwen2.5:3b

In [ ]:
answer = ask_ollama("qwen2.5:3b", prompt)
print(answer)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B")


In [ ]:
token_count = len(tokenizer.encode(prompt_2))
print(token_count)

In [ ]:
answer_1 = ask_ollama("qwen2.5:3b", prompt_2)

In [ ]:
print(answer_1)

In [ ]:
answer_3 = ask_ollama("qwen2.5:3b", prompt_3)
print(answer_3)

In [ ]:
answer_4= ask_ollama("qwen2.5:3b", prompt_4)
print(answer_4)

### Testing with Llama3.2:3b

In [ ]:
print(ask_ollama("llama3.2:3b", prompt))

In [ ]:
print(ask_ollama("llama3.2:3b", prompt_2))

In [ ]:
print(ask_ollama("llama3.2:3b", prompt_3))

In [ ]:
print(ask_ollama("llama3.2:3b", prompt_4))

### Testing with phi4-mini: 3.8b

In [ ]:
print(ask_ollama("phi4-mini:3.8b", prompt))

In [ ]:
print(ask_ollama("phi4-mini:3.8b", prompt_2))

In [ ]:
print(ask_ollama("phi4-mini:3.8b", prompt_3))

In [ ]:
print(ask_ollama("phi4-mini:3.8b", prompt_4))

## Evaluating on golden set

In [ ]:
def asking_ollama(model_name:str, prompt: str):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model":model_name,
                "prompt": prompt,
                "stream": False,
                "options": {
                            "temperature": 0.4
                }
            }
        )
    except Exception as e:
        return f"[REQUEST FAILED] could not reach Ollama: {e}"
    data = response.json()  
    #print(data.get("prompt_eval_count"), data.get("eval_count"))
    if response.status_code == 200:
        return data.get('response', "[NO RESPONSE FIELD IN REPLY]")
    else:
        return f"[OLLAMA ERROR {response.status_code}] {data.get('error', 'unknown error')}"

In [ ]:
def get_answer(models:list, question: str):
    prompt = create_prompt("prompt.md", question, example)
    final = {}
    details = {}
    for model in models:
        run = 0
        li = []
        
        while run<3:
            answer = asking_ollama(model, prompt)
            li.append({"run": run,
                       "answer": answer})
            run+=1
        details[f'{model}'] = li
    final[f'{question}'] = details
    return final        


In [ ]:
h = get_answer(["qwen2.5:3b", "llama3.2:3b"],"Under what conditions are medical devices manufactured and used within health institutions exempt from the requirements of the regulation?")
print(h)